# E7 Notebook 31 — G4 Risk Calibration and Abstention Audit

**This is a 20-model recovery/inference experiment, not a training experiment.** It restores the original private G3 LightGBM model ZIPs, checks their hashes, recalibrates **only on each frozen validation day**, then produces aggregate summaries on the previously inspected G3 test days.

**Research integrity:** The G3 test results have already been viewed. Therefore, this G4 experiment is strictly **RETROSPECTIVE EXPLORATORY**; G4 test scores cannot independently prove method superiority. The G4 protocol is locked in `G4_RISK_CALIBRATION_PROTOCOL_001.json`. Never retune based on these reused test results. An independently corrected source or new dataset is required for confirmatory claims.

M0: fixed 0.5 threshold from G3 B1. M1: validation-only alarm threshold under a 1% benign false-alarm rate bound using a simultaneous DKW empirical-CDF margin. M2: same alarm threshold plus a validation-only auto-benign low threshold, subject to a 5% attack-miss margin; intermediate predictions abstain. Bounds are calibration-day *i.i.d.-conditional diagnostics*, **not** cross-day guarantees.

Requirements: Kaggle Internet enabled, existing `dsshayan/dscicids` dataset attached, `pushRare` secret enabled; original 20 private G3 models must already be in the unpublished GitHub draft release. Nothing is retrained or published in a public model directory. Original source CSVs are never modified; no UNSW official test, IP/port/flow ID features, or CSV-neighbor chronology. Each complete aggregate report is saved to GitHub separately for resumability.

In [ ]:
from pathlib import Path
import hashlib,importlib.util,json,sys
from urllib.request import urlopen

REPO="itsCodeBakery/rare-attack-spatiotemporal-ids"
COMMIT="337cd629b857ed67d2888419e6a6428de93e75c7"
BASE="research/E7_CICIDS2017"
PINNED={
 "scripts/e7_g4_risk_calibration.py":"718bdae118e757ece719d5143447caff47c22b00",
 "scripts/e7_g4_publish.py":"d1472882a83ba617dd355e249947eaafde3259bb",
 "scripts/e7_g3_openworld_baseline.py":"f882154ff9757abe66a7fe588039eb26c9758170",
 "scripts/e7_g3_archive_v3.py":"033fbc2f7957f54bdaf7b6ad9ad6da6b26d8f619",
 "configs/G3_OPENWORLD_DAYHOLDOUT_PROTOCOL_001.json":"5ca597207bd8379860a3007b6cb7952efdde9b4f",
 "configs/G4_RISK_CALIBRATION_PROTOCOL_001.json":"e3099b9e792543f283a97172d44a1a53726a2dbd"
}
WORK=Path("/kaggle/working/E7_G4_RISK_ABSTENTION_001")
SNAPSHOT=WORK/"pinned_sources"
WORK.mkdir(parents=True,exist_ok=True)

def gitblob(content):
    return hashlib.sha1(b"blob "+str(len(content)).encode()+bytes([0])+content).hexdigest()

for name,expected in PINNED.items():
    target=SNAPSHOT/name
    if target.exists(): raw=target.read_bytes()
    else:
        with urlopen("https://raw.githubusercontent.com/"+REPO+"/"+COMMIT+"/"+BASE+"/"+name,timeout=90) as stream:
            raw=stream.read()
    assert gitblob(raw)==expected,"PINNED SOURCE MISMATCH: "+name
    if not target.exists():
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(raw)
def load(name,path):
    spec=importlib.util.spec_from_file_location(name,path)
    mod=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod
calibrate=load("g4_calibration",SNAPSHOT/"scripts/e7_g4_risk_calibration.py")
publish=load("g4_publisher",SNAPSHOT/"scripts/e7_g4_publish.py")
training=load("original_g3_training",SNAPSHOT/"scripts/e7_g3_openworld_baseline.py")
archive=load("original_g3_private_archive",SNAPSHOT/"scripts/e7_g3_archive_v3.py")
g3=json.loads((SNAPSHOT/"configs/G3_OPENWORLD_DAYHOLDOUT_PROTOCOL_001.json").read_text())
g4=json.loads((SNAPSHOT/"configs/G4_RISK_CALIBRATION_PROTOCOL_001.json").read_text())
training.check_protocol(g3)
assert g4["status"]=="LOCKED_BEFORE_G4_CALIBRATION"
assert g4["parent_protocol_blob"]==PINNED["configs/G3_OPENWORLD_DAYHOLDOUT_PROTOCOL_001.json"]
assert g4["constraints"]["seeds"]==[42,43,44,45,46]
assert calibrate.MAX_FPR==g4["constraints"]["validation_alarm_fpr_bound"]==0.01
assert calibrate.MAX_FNR==g4["constraints"]["validation_missed_attack_bound"]==0.05
assert calibrate.DELTA==g4["constraints"]["delta_for_each_one_sided_DKW"]
print("G4 PINNED PROTOCOL AND 6 SOURCE FILES VERIFIED")
print("SCIENTIFIC STATUS: RETROSPECTIVE EXPLORATORY — NO INDEPENDENT TEST CLAIMS")

## Private-model and original G3 evidence preflight

Before reading source data, this checks that the original G3 release remains **unpublished** and that all 20 original model assets and 20 G3 public metrics reports exist. We do not create another release or upload new models.

In [ ]:
import base64,hashlib,lightgbm as lgb
from kaggle_secrets import UserSecretsClient
from urllib.parse import quote
from urllib.error import HTTPError

token=UserSecretsClient().get_secret("pushRare")
assert token,"Enable Kaggle secret pushRare; never paste tokens into cells"
url=archive.ROOT+"/releases?per_page=100"
releases=archive.api("GET",url,token)
matches=[x for x in releases if x.get("tag_name")==archive.TAG]
assert len(matches)==1 and matches[0].get("draft") is True,(
    "Original G3 unpublished draft model release missing or not draft; STOP"
)
RELEASE=matches[0]
assets=archive.assets(RELEASE,token)
reports={}
for fold in g3["folds"]:
    fid=int(fold["fold"].rsplit("_",1)[-1])
    for seed in training.SEEDS:
        asset_name=f"G3_FOLD_{fid}_SEED_{seed}_PRIVATE.zip"
        assert asset_name in assets,"Previously trained private G3 model unavailable: "+asset_name
        path="research/E7_CICIDS2017/results/G3_OPENWORLD_DAYHOLDOUT/"+f"G3_FOLD_{fid}_SEED_{seed}.metrics.json"
        item=archive.api("GET",archive.ROOT+"/contents/"+quote(path,safe="/"),token)
        report=json.loads(base64.b64decode(item["content"]))
        archive.safe_metrics(report)
        assert report["fold"]==fold["fold"] and report["seed"]==seed
        assert report["test_day"]==fold["test_day"]
        assert report["validation_day"]==fold["validation_day"]
        assert report["protocol_blob_sha"]==PINNED["configs/G3_OPENWORLD_DAYHOLDOUT_PROTOCOL_001.json"]
        reports[(fid,seed)]={"data":report,"git_blob":item["sha"],"asset_name":asset_name}
print("PRIVATE G3 MODEL BACKUPS VERIFIED AVAILABLE: 20/20")
print("ORIGINAL G3 PUBLIC METRICS VERIFIED: 20/20")
print("NO NEW MODELS, PRIVATE RELEASES OR TRAINING REQUESTED")

## Original-source integrity and in-memory feature preparation

Reuses the exact strict CSV reader and derived blank-row exclusion from frozen G3, with original SHA256 verification and whole-day source grouping. Dataset arrays remain in Kaggle memory and are never uploaded publicly.

In [ ]:
DATA_ROOT=Path("/kaggle/input/datasets/dsshayan/dscicids")
by_day,features,original_provenance=training.build_all(DATA_ROOT,g3,chunk_size=25000)
feature_digest=hashlib.sha256(json.dumps(features,ensure_ascii=False).encode()).hexdigest()
assert len(features)>=65
assert sum(len(v["y"]) for v in by_day.values())==2830743
hashes={s["source_file"]:s["source_file_sha256"] for s in g3["source_files"]}
assert len(hashes)==8
for fold in g3["folds"]:
    for seed in training.SEEDS:
        src=reports[(int(fold["fold"].rsplit("_",1)[-1]),seed)]["data"]
        assert src["features_sha256"]==feature_digest
        assert src["source_hashes"]==hashes
print("G4 DATA PREFLIGHT PASSED:",sum(len(v["y"]) for v in by_day.values()),"populated source rows")
print("ALL ORIGINAL DATA HASHES / FEATURE IDENTITIES AGREE WITH G3")

## Restore-and-calibrate, one checkpoint at a time

For each (fold, seed): restore its private original ZIP, verify SHA256 and model metadata, perform **validation-only** G4 calibration, then score the previously inspected G3 test day for descriptive analysis. Immediately publish an aggregate G4 report. Any existing report is validated and skipped. No training is performed.

In [ ]:
import gc,numpy as np
from pathlib import Path
RESULTS=WORK/"verified_reports"
RESULTS.mkdir(parents=True,exist_ok=True)
saved=[]
def get_g4_previous(fid,seed):
    name=f"G4_FOLD_{fid}_SEED_{seed}.json"
    link=publish.ROOT+"/contents/"+quote(publish.PREFIX+"/"+name,safe="/")
    try:obj=archive.api("GET",link,token)
    except HTTPError as e:
        if e.code==404:return None
        raise
    data=json.loads(base64.b64decode(obj["content"]))
    publish.validate(data)
    return data

for fold in g3["folds"]:
    fid=int(fold["fold"].rsplit("_",1)[-1])
    v=by_day[fold["validation_day"]]
    test=by_day[fold["test_day"]]
    print("\n"+"="*68)
    print("EXPLORATORY G4",fold["fold"],"val",fold["validation_day"],"test",fold["test_day"],flush=True)
    for seed in training.SEEDS:
        prior=reports[(fid,seed)]
        orig=prior["data"]
        existing=get_g4_previous(fid,seed)
        if existing is not None:
            assert existing["g4_protocol_git_blob"]==PINNED["configs/G4_RISK_CALIBRATION_PROTOCOL_001.json"]
            assert existing["original_g3_metrics_blob"]==prior["git_blob"]
            assert existing["original_model_sha256"]==orig["model_asset_sha256"]
            assert existing["features_sha256"]==feature_digest
            assert existing["scientific_status"]=="RETROSPECTIVE_EXPLORATORY_REUSED_G3_TEST"
            saved.append(existing)
            print("G4",fid,seed,"ORIGINAL VERIFIED ON GITHUB — SKIPPED",flush=True)
            continue
        zipdata=archive.get_private_asset(assets[prior["asset_name"]],token)
        assert archive.sha(zipdata)==orig["model_asset_sha256"],"PRIVATE MODEL SHA256 mismatch"
        required={"schema":"E7_G3_PRIVATE_MODEL_001","fold":fold["fold"],"seed":seed,
          "protocol_blob_sha":PINNED["configs/G3_OPENWORLD_DAYHOLDOUT_PROTOCOL_001.json"],
          "features_sha256":feature_digest,"feature_names":features,
          "train_days":fold["train_days"],"validation_day":fold["validation_day"],
          "test_day":fold["test_day"]}
        model,meta=archive.unpack_model(zipdata,required)
        assert meta["lightgbm_version"]==lgb.__version__
        assert meta["hyperparameters"]==orig["hyperparameters"]
        probability_validation=model.predict(v["X"],num_threads=4)
        # Old G3 val selection is verified as a model/feature identity check;
        # it NEVER directs G4 threshold tuning.
        original_choice=training.choose_threshold(v["y"],probability_validation)
        assert original_choice==orig["validation_selection"],"G3 model calibration identity drift"
        selection=calibrate.choose_band(v["y"],probability_validation)
        assert selection["alarm_high"]>selection["auto_benign_low"]
        # G4 decisions fully frozen BEFORE computing any G3 test probabilities.
        probability_test=model.predict(test["X"],num_threads=4)
        m0_check=training.evaluate(test["y"],probability_test,0.5)
        assert abs(m0_check["macro_f1"]-orig["b1"]["macro_f1"])<1e-8,(
            "Restored G3 prediction identity mismatch; do not publish G4"
        )
        m1=training.evaluate(test["y"],probability_test,
                             selection["alarm_high"],test["labels"])
        m2=calibrate.selective_metrics(test["y"],probability_test,
                                       test["labels"],selection)
        record={
         "schema":"E7_G4_CALIBRATION_REPORT_001",
         "experiment":g4["protocol_id"],"parent_experiment":g3["protocol_id"],
         "g4_protocol_git_blob":PINNED["configs/G4_RISK_CALIBRATION_PROTOCOL_001.json"],
         "g3_protocol_git_blob":PINNED["configs/G3_OPENWORLD_DAYHOLDOUT_PROTOCOL_001.json"],
         "fold":fold["fold"],"seed":seed,"train_days":fold["train_days"],
         "validation_day":fold["validation_day"],"test_day":fold["test_day"],
         "original_source_hashes":hashes,"original_model_sha256":orig["model_asset_sha256"],
         "features_sha256":feature_digest,"original_g3_metrics_blob":prior["git_blob"],
         "original_label_provenance":"UNVERIFIED_ORIGINAL_CICIDS2017_LABELS",
         "validation_only_selection":selection,
         "m0_g3_b1_macro_f1":orig["b1"]["macro_f1"],
         "m1_alarm_metrics":m1,"m2_selective_metrics":m2,
         "scientific_status":"RETROSPECTIVE_EXPLORATORY_REUSED_G3_TEST",
         "limitations":[
             "G3 test-day outcomes were previously inspected; G4 comparisons are retrospective exploratory, not unbiased confirmation",
             "Validation-day DKW bands assume independent identically distributed flows; no distribution-free test-day risk guarantee",
             "Original CIC-IDS2017 class annotations remain uncorrected/provisional",
             "Attack families and capture-day environmental conditions are confounded",
             "No verified temporal sequence; no new classifier fitted; no frozen UNSW retest"
         ]}
        status=publish.publish(record,api=archive.api,token=token,output_dir=RESULTS)
        saved.append(record)
        print("G4 VERIFIED:",fid,seed,status,"M1 FPR",round(m1["fpr"],4),
              "M1 recall",round(m1["recall_attack"],4),
              "M2 coverage",round(m2["coverage"],4),
              "M2 selective error",m2["selective_error"],flush=True)
        del model,zipdata,probability_validation,probability_test
        gc.collect()
assert len(saved)==20
print("G4 VERIFIED REPORTS: 20/20")

## Publish the descriptive four-fold summary

Every value below comes from the individually verified original G4 reports. No threshold or model changes are made after observing test outcomes.

In [ ]:
summary_by={}
for fold in g3["folds"]:
    r=[z for z in saved if z["fold"]==fold["fold"]]
    assert sorted(z["seed"] for z in r)==list(training.SEEDS)
    def avg(metric,field):
        return float(np.mean([x[metric][field] for x in r]))
    def avg_nullable(metric,field):
        v=[x[metric][field] for x in r if x[metric][field] is not None]
        return float(np.mean(v)) if v else None
    summary_by[fold["fold"]]={
       "test_day":fold["test_day"],"n_seeds":len(r),
       "g3_b1_macro_f1_mean":float(np.mean([z["m0_g3_b1_macro_f1"] for z in r])),
       "m1_macro_f1_mean":avg("m1_alarm_metrics","macro_f1"),
       "m1_attack_recall_mean":avg("m1_alarm_metrics","recall_attack"),
       "m1_fpr_mean":avg("m1_alarm_metrics","fpr"),
       "m2_coverage_mean":avg("m2_selective_metrics","coverage"),
       "m2_selective_error_mean_on_nonempty_decisions":avg_nullable("m2_selective_metrics","selective_error"),
       "m2_test_benign_fpr_mean":avg("m2_selective_metrics","test_benign_fpr"),
       "m2_test_attack_alert_recall_mean":avg("m2_selective_metrics","test_attack_alert_recall"),
       "m1_validation_feasible_seeds":sum(bool(z["validation_only_selection"]["alarm_bound_feasible"]) for z in r),
       "m2_validation_feasible_seeds":sum(bool(z["validation_only_selection"]["benign_bound_feasible"]) for z in r)}
summary={"schema":"E7_G4_SUMMARY_001","experiment":g4["protocol_id"],
  "g4_protocol_git_blob":PINNED["configs/G4_RISK_CALIBRATION_PROTOCOL_001.json"],
  "n_reports":20,"n_folds":4,"n_seeds":5,"fold_results":summary_by,
  "scientific_status":"RETROSPECTIVE_EXPLORATORY_REUSED_G3_TEST",
  "limitations":[
    "Already inspected G3 test days, therefore G4 scores cannot support confirmatory superiority claims",
    "Validation-calibrated false-positive and missed-attack bounds do not transfer automatically across capture days",
    "Source labels are original uncorrected CIC-IDS2017; independent corrected-data confirmation required",
    "Abstention coverage and error are reported together; never omit rejected flows silently"
  ]}
out=publish.publish_summary(summary,api=archive.api,token=token,output_dir=RESULTS)
print("G4 SUMMARY:",out)
print("PUBLIC G4 REPORTS: https://github.com/"+REPO+"/tree/main/"+publish.PREFIX)
print("EXPLORATORY ONLY: no new model fitting, no confirmed cross-day bound.")
print("Please share this output for the G4 scientific interpretation.")